In [7]:
import pandas as pd

from xgboost import XGBRegressor
import lightgbm as lgb

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.metrics import r2_score

In [8]:
# Preset Values

filename = "CRMLS_0525-0526_enriched.csv"

main_cols = ["BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet",
             "DaysOnMarket", "YearBuilt", "PostalCode", "SaleMonth"]

extras = ["ViewYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN"]
extra_cols = [a+"_True" for a in extras] + [a+"_False" for a in extras]

totals = main_cols+extra_cols

target = "ClosePrice"

In [9]:
def load_df(file=filename):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  df["CloseDate"] = pd.to_datetime(df["CloseDate"])     # Converts "CloseDate" values to datetime type
  return df

In [16]:
main_df = load_df()

# **Gradient Boosting**



In [10]:
def test_train_split(df):
  """
  Takes a DataFrame
  Encodes "PropertyType" column
  Returns a defined training and test set for the DataFrame
  """
  yr_mo = []
  for i in df['CloseDate']:                                         # For each date in the "CloseDate" column
    yr, mo = i.year, i.month                                          # Define the year and month values of date i
    yr_mo.append([yr,mo])                                             # Append to "yr_mo" a list of date i's year and month
  te_set = [b for b in range(len(yr_mo)) if yr_mo[b] == [2026,5]]   # Define a list of row #s with date 05/2026
  te_rng = te_set[0::len(te_set)-1]                                 # Define a list of the first and last row in "te_set"
  tr, te = df[0:te_rng[0]], df[te_rng[0]:te_rng[1]]             # Define the training and test sets of the inputted df

  return tr, te

**XGBoost**

In [22]:
def XGB(tr, te, feat=totals, targ=target):
  """
  Takes a training DataFrame and test DataFrame
  """
  x_tr = tr[feat].values            # Define the x_train set values
  y_tr = tr[targ].values            # Define the y_train set values
  x_te = te[feat].values            # Define the x_test set values
  y_te = te[targ].values            # Define the y_test set values

  model = XGBRegressor(random_state=0)
  # max_depth=kwargs.dep, learning_rate=kwargs.l_rate, n_estimators=kwargs.est)        # Define Decision Tree Regression model
  model.fit(x_tr, y_tr)             # Fit training data to model
  y_pred = model.predict(x_te)      # Models the predicted y values from x_test values

  r2 = r2_score(y_te, y_pred)       # Computes the r2 score of y_test and the predicted y

  return r2

In [23]:
def main():
  train, test = test_train_split(main_df)

  r2_scores = []
  for col in totals:
    r2s = XGB(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = XGB(train, test, cols)
    print(f"Correlation of {cols}: \t\t {round(r2s,4)}")

In [24]:
if __name__ == "__main__":
  main()

Correlation of ['LivingArea']: 		 0.281
Correlation of ['LivingArea', 'BathroomsTotalInteger']: 		 0.1937
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode']: 		 -0.0026
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal']: 		 -1.4972
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False']: 		 -0.1645
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 -0.4637
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 -0.521
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'LotSizeSquareFeet']: 		 -2.8139
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False',

***Results***

Highest R2 Score:  **0.281**

* Included Features:

  - ['LivingArea']

**Gradient Boosting Regressor**

In [25]:
def GBR(tr, te, feat=totals, targ=target):
  """
  Takes a training DataFrame and test DataFrame
  """
  x_tr = tr[feat].values            # Define the x_train set values
  y_tr = tr[targ].values            # Define the y_train set values
  x_te = te[feat].values            # Define the x_test set values
  y_te = te[targ].values            # Define the y_test set values

  model = GradientBoostingRegressor(random_state=0)
# max_depth=dep, learning_rate=l_rate, n_estimators=est)        # Define Decision Tree Regression model
  model.fit(x_tr, y_tr)             # Fit training data to model
  y_pred = model.predict(x_te)      # Models the predicted y values from x_test values

  r2 = r2_score(y_te, y_pred)       # Computes the r2 score of y_test and the predicted y

  return r2

In [26]:
def main():
  train, test = test_train_split(main_df)

  r2_scores = []
  for col in totals:
    r2s = GBR(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = GBR(train, test, cols)
    print(f"Correlation of {cols}: \t\t {round(r2s,4)}")

In [27]:
if __name__ == "__main__":
  main()

Correlation of ['LivingArea']: 		 0.3028
Correlation of ['LivingArea', 'BathroomsTotalInteger']: 		 0.2668
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode']: 		 0.3994
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal']: 		 0.4233
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False']: 		 0.3879
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 0.4007
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.3985
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True']: 		 0.4036
Correlation of ['LivingArea', 'BathroomsTotalInteger', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'F

***Results***

Highest R2 Score:  **0.443**

* Used all columns in features list

---

* Less wide-ranging values of R2 scores than XGBoost

**LightGBM**

In [34]:
def L_GBM(tr, te, feat=totals, targ=target, dep=10, l_rate=10, est=10):
  """
  Takes a training DataFrame and test DataFrame
  """
  x_tr, y_tr = tr[feat].values, tr[targ].values       # Define the x_train, y_train set values
  x_te, y_te = te[feat].values, te[targ].values       # Define the x_test, y_test set values

  lgb_tr = lgb.Dataset(x_tr, y_tr, free_raw_data=False)
  lgb_te = lgb.Dataset(x_te, y_te, free_raw_data=False)

  model = lgb.train(params={"max_depth": dep, "learning_rate": l_rate,
                            "n_estimators": est}, train_set=lgb_tr, valid_sets=lgb_tr)        # Define Decision Tree Regression model
  y_pred = model.predict(x_te, num_iteration=model.best_iteration)      # Models the predicted y values from x_test values

  r2 = r2_score(y_te, y_pred)       # Computes the r2 score of y_test and the predicted y

  return r2

In [32]:
def main():
  train, test = test_train_split(main_df)

  r2_scores = []
  for col in totals:
    r2s = L_GBM(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = L_GBM(train, test, cols)
    print(f"Correlation of {cols}: \t\t {round(r2s,4)}")

In [35]:
if __name__ == "__main__":
  main()

[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=10) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=1024) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=10) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=1024) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015099 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15
[LightGBM] [Info] Number of data points in the train set: 92922, number of used features: 1
[LightGBM] [Info] Start training from score 1293408.040061
[LightGBM] [Warning] No further splits with pos

**Histogram-based Gradient Boosting Regressor**

In [37]:
def HGBR(tr, te, feat=totals, targ=target):
  """
  Takes a training DataFrame and test DataFrame
  """
  x_tr = tr[feat].values            # Define the x_train set values
  y_tr = tr[targ].values            # Define the y_train set values
  x_te = te[feat].values            # Define the x_test set values
  y_te = te[targ].values            # Define the y_test set values

  model = HistGradientBoostingRegressor(random_state=0)
  # max_depth=dep, learning_rate=l_rate)             # Define Decision Tree Regression model
  model.fit(x_tr, y_tr)             # Fit training data to model
  y_pred = model.predict(x_te)      # Models the predicted y values from x_test values

  r2 = r2_score(y_te, y_pred)       # Computes the r2 score of y_test and the predicted y

  return r2

In [38]:
def main():
  train, test = test_train_split(main_df)

  r2_scores = []
  for col in totals:
    r2s = HGBR(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = HGBR(train, test, cols)
    print(f"Correlation of {cols}: \t\t {round(r2s,4)}")

In [39]:
if __name__ == "__main__":
  main()

Correlation of ['BathroomsTotalInteger']: 		 0.278
Correlation of ['BathroomsTotalInteger', 'LivingArea']: 		 0.2792
Correlation of ['BathroomsTotalInteger', 'LivingArea', 'PostalCode']: 		 0.3072
Correlation of ['BathroomsTotalInteger', 'LivingArea', 'PostalCode', 'BedroomsTotal']: 		 0.3261
Correlation of ['BathroomsTotalInteger', 'LivingArea', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False']: 		 0.3386
Correlation of ['BathroomsTotalInteger', 'LivingArea', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 0.3221
Correlation of ['BathroomsTotalInteger', 'LivingArea', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.3221
Correlation of ['BathroomsTotalInteger', 'LivingArea', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'LotSizeSquareFeet']: 		 0.2544
Correlation of ['BathroomsTotalInteger', 'LivingArea', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_F

***Results***

Highest R2 Score:  **0.3386**

* Included Features:

  - ['BathroomsTotalInteger', 'LivingArea', 'PostalCode', 'BedroomsTotal', 'PoolPrivateYN_False']

* Excluded Features:

  - ['FireplaceYN_False', 'FireplaceYN_True', 'LotSizeSquareFeet', 'PoolPrivateYN_True', 'ViewYN_True', 'ViewYN_False', 'YearBuilt', 'NewConstructionYN_True', 'SaleMonth', 'NewConstructionYN_False']

---

* Lower values than Gradient Boosting Regressor

# **Hyperparameter Tuning**

In [ ]:
def main():
  train, test = test_train_split(main_df)

  r2_dict = {"R2_score": [], "max_depth": [], "learning_rate": [], "n_estimators": [], "columns": []}


  # XGBoost

  r2_scores = []
  for col in totals:
    r2s = XGB(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  r2_cols = []
  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = XGB(train, test, cols)
    r2_cols.append([cols, r2s])
  r2_cols = sorted(r2_cols, key=lambda x: x[1], reverse=True)

  r2_scrs = []
  for depth in range(0,26):
    for learn_rate in range(0,26):
      for n_est in range(0,26):
        r2 = XGB(train, test, feat=r2_cols[0][0], dep=depth, l_rate=learn_rate, est=n_est)
        all = [r2, depth, learn_rate, n_est, r2_cols[0][0]]
        r2_scrs.append(all)
  r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
  for i in range(len(r2_scrs[0])):
    r2_dict[r2_dict.keys()[i]].append(r2_scrs[0][i])


  # Gradient Boosting Regressor

  r2_scores = []
  for col in totals:
    r2s = GBR(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  r2_cols = []
  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = GBR(train, test, cols)
    r2_cols.append([cols, r2s])
  r2_cols = sorted(r2_cols, key=lambda x: x[1], reverse=True)

  r2_scrs = []
  for depth in range(0,26):
    for learn_rate in range(0,26):
      for n_est in range(0,26):
        r2 = GBR(train, test, feat=r2_cols[0][0], dep=depth, l_rate=learn_rate, est=n_est)
        all = [r2, depth, learn_rate, n_est, r2_cols[0][0]]
        r2_scrs.append(all)
  r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
  for i in range(len(r2_scrs[0])):
    r2_dict[r2_dict.keys()[i]].append(r2_scrs[0][i])


  # LightGBM

  r2_scores = []
  for col in totals:
    r2s = L_GBM(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  r2_cols = []
  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = L_GBM(train, test, cols)
    r2_cols.append([cols, r2s])
  r2_cols = sorted(r2_cols, key=lambda x: x[1], reverse=True)

  r2_scrs = []
  for depth in range(0,26):
    for learn_rate in range(0,26):
      for n_est in range(0,26):
        r2 = L_GBM(train, test, feat=r2_cols[0][0], dep=depth, l_rate=learn_rate, est=n_est)
        all = [r2, depth, learn_rate, n_est, r2_cols[0][0]]
        r2_scrs.append(all)
  r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
  for i in range(len(r2_scrs[0])):
    r2_dict[r2_dict.keys()[i]].append(r2_scrs[0][i])


  # Histogram-based Gradient Boosting Regressor

  r2_scores = []
  for col in totals:
    r2s = HGBR(train, test, [col])
    r2_scores.append(r2s)
  r2_lst = [[totals[i], r2_scores[i]] for i in range(len(r2_scores))]
  r2_lst = sorted(r2_lst, key=lambda x: x[1], reverse=True)

  r2_cols = []
  for i in range(1, len(r2_scores)):
    maximum = i
    cols = [r2_lst[a][0] for a in range(maximum)]
    r2s = HGBR(train, test, cols)
    r2_cols.append([cols, r2s])
  r2_cols = sorted(r2_cols, key=lambda x: x[1], reverse=True)

  r2_scrs = []
  for depth in range(0,26):
    for learn_rate in range(0,26):
      for n_est in range(0,26):
        r2 = HGBR(train, test, feat=r2_cols[0][0], dep=depth, l_rate=learn_rate, est=n_est)
        all = [r2, depth, learn_rate, n_est, r2_cols[0][0]]
        r2_scrs.append(all)
  r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
  for i in range(len(r2_scrs[0])):
    r2_dict[r2_dict.keys()[i]].append(r2_scrs[0][i])


  result_df = pd.DataFrame(r2_dict)
  return result_df

In [ ]:
if __name__ == "__main__":
  main()